In [1]:
import tensorflow as tf
import keras
from keras import layers
import pickle
import numpy as np
import pandas as pd
from keras.models import load_model

from scipy.stats import norm
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import ks_2samp, anderson, wasserstein_distance
from sklearn.metrics import r2_score
import glob

import warnings
warnings.filterwarnings("ignore")

In [2]:
lat = np.arange(-90, 90, 180/192)
lon = np.arange(0, 360, 360/288)

lat_bnd1 = np.where(lat==10.3125)[0][0]
lat_bnd2 = np.where(lat==50.625)[0][0]
lon_bnd1 = np.where(lon==210)[0][0]
lon_bnd2 = np.where(lon==300)[0][0]

In [3]:
#set the base path 
base_path = '/sfs/weka/scratch/zkq5md'

#load the dataframe
df_se_era5 = pd.read_csv(f'{base_path}/Data/ERA5/Final/SE_detrend_precip.csv', index_col='Unnamed: 0')
df_gp_era5 = pd.read_csv(f'{base_path}/Data/ERA5/Final/GP_detrend_precip.csv', index_col='Unnamed: 0')
df_ca_era5 = pd.read_csv(f'{base_path}/Data/ERA5/Final/CA_detrend_precip.csv', index_col='Unnamed: 0')
df_se_era5

,Datetimes,Maximum Precipitation,lats,lons
0,1950-01-01,29.729749,32.8125,266.25
1,1950-01-02,39.121912,35.6250,270.00
2,1950-01-03,18.399313,35.6250,266.25
3,1950-01-04,59.563390,35.6250,267.50
4,1950-01-05,40.699897,35.6250,271.25
...,...,...,...,...
9977,2015-12-27,75.272000,34.6875,266.25
9978,2015-12-28,51.817838,35.6250,267.50
9979,2015-12-29,26.021470,34.6875,276.25
9980,2015-12-30,44.338429,30.9375,271.25


In [4]:
with open(f'{base_path}/Data/ERA5/Final/X_era5.pkl', "rb") as f:
    x = pickle.load(f)
    
x = np.transpose(np.array(x), (0, 2, 3, 1))

x = x[:, lat_bnd1:lat_bnd2, lon_bnd1:lon_bnd2, :]

x_test_era5 = x[8000:]

y_se_era5_test = df_se_era5['Maximum Precipitation'].values[8000:]
y_gp_era5_test = df_gp_era5['Maximum Precipitation'].values[8000:]
y_ca_era5_test = df_ca_era5['Maximum Precipitation'].values[8000:]



In [5]:
polyfit_se = pd.read_csv(f'{base_path}/Data/ERA5/Final/SE_poltfit.csv', index_col='Unnamed: 0')
polyfit_gp = pd.read_csv(f'{base_path}/Data/ERA5/Final/GP_poltfit.csv', index_col='Unnamed: 0')
polyfit_ca = pd.read_csv(f'{base_path}/Data/ERA5/Final/CA_poltfit.csv', index_col='Unnamed: 0')

In [6]:
#set the base path 
base_path = '/sfs/weka/scratch/zkq5md'

with open(f"{base_path}/Full_Domain/X_test.pkl", "rb") as f:
    x_test_cesm = pickle.load(f)
    
x_test_cesm = x_test_cesm[:, lat_bnd1:lat_bnd2, lon_bnd1:lon_bnd2, :]
    
print(x_test_cesm.shape)

(22652, 43, 72, 3)


In [7]:
#the set the validata and testing members
val_num = 15
test_num = val_num +2

#set the range of member numbers
val_nums = np.arange(val_num,val_num+2,1)
test_nums = np.arange(test_num,test_num+2,1)

In [8]:
#load the dataframe
df_se_cesm = pd.read_csv(f'{base_path}/Full_Domain/SE_detrend_precip.csv', index_col='Unnamed: 0')
df_gp_cesm = pd.read_csv(f'{base_path}/Full_Domain/GP_detrend_precip.csv', index_col='Unnamed: 0')
df_ca_cesm = pd.read_csv(f'{base_path}/Full_Domain/CA_detrend_precip.csv', index_col='Unnamed: 0')

df_se_cesm=df_se_cesm[(df_se_cesm['Member'].isin(test_nums))].copy()
df_gp_cesm=df_gp_cesm[(df_gp_cesm['Member'].isin(test_nums))].copy()
df_ca_cesm=df_ca_cesm[(df_ca_cesm['Member'].isin(test_nums))].copy()

y_se_cesm_test = df_se_cesm['Maximum Precipitation'].values
y_gp_cesm_test = df_gp_cesm['Maximum Precipitation'].values
y_ca_cesm_test = df_ca_cesm['Maximum Precipitation'].values

df_se_cesm

,Datetimes,Member,Maximum Precipitation,lats,lons
181216,1940-01-01 00:00:00,17.0,46.601204,30.628272,280.00
181217,1940-01-02 00:00:00,17.0,8.604802,35.340314,278.75
181218,1940-01-03 00:00:00,17.0,53.491662,35.340314,276.25
181219,1940-01-04 00:00:00,17.0,41.651847,35.340314,276.25
181220,1940-01-05 00:00:00,17.0,16.084387,35.340314,276.25
...,...,...,...,...,...
203863,2014-12-28 00:00:00,18.0,12.462667,35.340314,266.25
203864,2014-12-29 00:00:00,18.0,9.678012,35.340314,272.50
203865,2014-12-30 00:00:00,18.0,39.065999,34.397906,277.50
203866,2014-12-31 00:00:00,18.0,18.523190,35.340314,280.00


In [9]:
se_polyfit = pd.read_csv(f'{base_path}/Full_Domain/SE_polyfit_slope_int.csv', index_col='Unnamed: 0')
gp_polyfit = pd.read_csv(f'{base_path}/Full_Domain/GP_polyfit_slope_int.csv', index_col='Unnamed: 0')
ca_polyfit = pd.read_csv(f'{base_path}/Full_Domain/CA_polyfit_slope_int.csv', index_col='Unnamed: 0')

In [ ]:
locs = ['se', 'gp', 'ca']
dirs = ['pre', 'pre', 'post', 'brute']
datasets = ['cesm', 'era5', 'era5', 'era5']
model_types = ['mse', 'weight_mse', 'weight_mae', 'line']

for num1, direct in enumerate(dirs):
    
    for num2, loc in enumerate(locs):

        for model_type in model_types:

            model_list = glob.glob(f'{base_path}/Data/ERA5/models/trans_learn/{direct}/{loc}_max_precip_{model_type}_model*')

            for num3, model_path in enumerate(model_list):

                if model_type == 'weight_mse' or model_type == 'line':
                    model = load_model(f"{model_path}", compile=False)

                else:
                    model = load_model(f"{model_path}", safe_mode=False)

                locals()[f'y_pred{num3+1}'] = model.predict(locals()[f'x_test_{datasets[num1]}'], verbose=0).flatten()
            locals()[f'df_{direct}_{datasets[num1]}_{loc}_{model_type}'] = pd.DataFrame({'Y_max_test': locals()[f'y_{loc}_{datasets[num1]}_test'], 'y_pred1': y_pred1,
                                                                        'y_pred2': y_pred2, 'y_pred3': y_pred3, 'y_pred4': y_pred4,
                                                                        'y_pred5': y_pred5, 'y_pred6': y_pred6, 'y_pred7': y_pred7,
                                                                        'y_pred8': y_pred8, 'y_pred9': y_pred9, 'y_pred10': y_pred10})

I0000 00:00:1768419568.261553  621444 service.cc:146] XLA service 0x7f478c008500 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768419568.261618  621444 service.cc:154]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1768419568.530355  621444 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
def cesm_retrend(vals, poly):
    
    poly1 = np.arange(vals[0:int(vals.shape[0]/2)].shape[0])
    poly2 = np.arange(vals[0:int(vals.shape[0]/2)].shape[0])

    poly1 = poly.iloc[16].Member * poly1 + poly.iloc[16].Intercepts
    poly2 = poly.iloc[17].Member * poly2 + poly.iloc[17].Intercepts

    y1 = poly1 + vals[0:int(vals.shape[0]/2)]
    y2 = poly2 + vals[int(vals.shape[0]/2):]

    vals = np.concatenate(np.array([y1, y2]), axis=0)
    vals[vals < 0] = 0
    
    return vals

In [ ]:
def era5_retrend(vals, poly):
    
    poly1 = np.arange(8000, 8000+vals[0:int(vals.shape[0])].shape[0], 1)
    poly1 = poly.iloc[0].Slopes * poly1 + poly.iloc[0].Intercept

    y = poly1 + vals[0:int(vals.shape[0])]

    y[y < 0] = 0

    return y

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.transforms import Bbox
import numpy as np

# 1. Expand Figure and GridSpec
# Added one more row (7 total) and one more 0.2 ratio for the second spacer
fig = plt.figure(figsize=(21.5, 34.5)) 
gs = gridspec.GridSpec(7, 3, height_ratios=[1, 0.2, 1, 1, 1, 0.2, 1.2]) 

columns_to_analyze = ['y_pred1', 'y_pred2', 'y_pred3', 'y_pred4', 'y_pred5', 
                      'y_pred6', 'y_pred7', 'y_pred8', 'y_pred9', 'y_pred10']
regions = ['Southeast', 'Great Plains', 'California']
info = ['CESM2-LE Trained', 'CESM2-LE Trained', 'CESM2-LE Trained \n  Tuned on ERA5', 'ERA5 Trained', 'Regional Stats']
abc = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O']
model_types = ['mse', 'weight_mse', 'weight_mae', 'line']

j = 0
axes = [] 

# --- PART 1: Your Original Data Loop ---
for num1, direct in enumerate(dirs):
    for num2, loc in enumerate(locs):
        row = j // 3
        col = j % 3
        
        # Original logic: Skip the first spacer at grid row 1
        if row >= 1:
            ax = plt.subplot(gs[row + 1, col])
        else:
            ax = plt.subplot(gs[row, col])
        
        axes.append(ax)
        j += 1

        for model_type in model_types:
            df = locals()[f'df_{direct}_{datasets[num1]}_{loc}_{model_type}']
            vals = df[columns_to_analyze].mean(axis=1)

            if datasets[num1] == 'cesm':
                vals = cesm_retrend(vals, locals()[f'{loc}_polyfit'])
                y_max = cesm_retrend(locals()[f'y_{loc}_{datasets[num1]}_test'], locals()[f'{loc}_polyfit'])
            else:
                vals = era5_retrend(vals, locals()[f'polyfit_{loc}'])
                y_max = era5_retrend(locals()[f'y_{loc}_{datasets[num1]}_test'], locals()[f'polyfit_{loc}'])

            ecdf = stats.ecdf(vals)
            ax.plot(ecdf.cdf.quantiles, np.log10(1 - ecdf.cdf.probabilities), lw=3)

            locals()[f'r_{model_type}_{loc}'] = r2_score(y_max, vals)

        ecdf = stats.ecdf(y_max)
        ax.plot(ecdf.cdf.quantiles, np.log10(1 - ecdf.cdf.probabilities), lw=4, color='k')

        ax.legend(['MSE', 'Weighted MSE', 'Weighted MAE', 'Linearized CNN', 'Ground Truth'], fontsize=15, loc='upper right')
        ax.set_xlabel('Precipitation (mm/Day)', fontsize=22)
        ax.set_ylabel('Exceedance Probability', fontsize=22)
        ax.set_xticks(np.arange(0, 150, 20))
        ax.set_xticklabels(np.arange(0, 150, 20), fontsize=15)
        ax.set_yticks(np.arange(0, -5, -1))
        ax.set_yticklabels(['$10^{0}$', '$10^{-1}$', '$10^{-2}$', '$10^{-3}$', '$10^{-4}$'], fontsize=15)
        ax.grid()

        if j <= 3:
            ax.set_title(f'{regions[num2]}', fontsize=40, fontweight='bold')
        if num2 == 0:
            #ax.set_ylabel(f'{info[num1]}', fontsize=27, fontweight='bold')
            ax.annotate(f'{info[num1]}', xy=(-.37,.1*[1,1,1,2][row]), xycoords='axes fraction', 
                        fontsize=25, rotation='vertical', fontweight='bold')

        ax.annotate(f'$R^2$: {str(locals()[f'r_mse_{loc}'] )[:4]}', (25, -4), textcoords="offset points",
            xytext=(0,65), ha='center', color='blue', fontsize=20, fontweight='bold')

        ax.annotate(f'$R^2$: {str(locals()[f'r_weight_mse_{loc}'] )[:4]}', (25, -4), textcoords="offset points",
                    xytext=(0,45), ha='center', color='orange', fontsize=20, fontweight='bold')

        ax.annotate(f'$R^2$: {str(locals()[f'r_weight_mae_{loc}'] )[:4]}', (25, -4), textcoords="offset points",
                    xytext=(0,25), ha='center', color='green', fontsize=20, fontweight='bold')

        ax.annotate(f'$R^2$: {str(locals()[f'r_line_{loc}'] )[:4]}', (25, -4), textcoords="offset points",
                    xytext=(0,5), ha='center', color='red', fontsize=20, fontweight='bold')
        
        ax.annotate(f'{abc[j-1]}.)', (25, 0), textcoords="offset points",  color='k', fontsize=20,
                    xytext=(-80, 20 + np.max(np.log10(1 - ecdf.cdf.probabilities))), ha='center',
                    bbox=dict(boxstyle='round', facecolor='w', alpha=1), fontweight='bold')


# --- PART 2: The New Spacer & Box Plot Row (Fake Data) ---
# We use grid row 6 (after the second spacer at index 5)
for num2, loc in enumerate(locs):
    ax = plt.subplot(gs[6, num2])
    axes.append(ax)
    
    data =[]
    for num4 in range(locals()[f'x_test_{datasets[num1]}'].shape[-1]):
        row = []
        for num1, direct in enumerate(dirs):
            for model_type in model_types:
                for num3, model_path in enumerate(model_list):
                    df = pd.read_csv(f'df_{direct}_{datasets[num1]}_{loc}_{model_type}_{num4}')
                    
                    for model_num in np.arange(1,11,1):
                        row.append(r2_score(df['Y_max_test'].values, df[f'y_pred{model_num}'].values))
                        
        data.append(row)
    print(np.max(np.array(data), axis = 1))
    
    bp = ax.boxplot(data, patch_artist=True, labels=['CWV', 'Z500', 'TAS'])
    ax.axhline(y=np.mean(np.array([locals()[f'r_mse_{loc}'], locals()[f'r_weight_mse_{loc}'], locals()[f'r_weight_mae_{loc}']])), color='blue')
    custom_line = Line2D([0], [0], color='blue', lw=2, label='Average nonzeroed $R^2$ value')
    ax.legend(handles=[custom_line], fontsize=15, loc='upper right')
    
    
    # Styling to match your model colors
    colors = ['blue', 'orange', 'green']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    
    # Labeling (e.g., M, N, O)
    ax.annotate(f'{abc[len(axes)-1]}.)', (0.05, 0.9), xycoords='axes fraction', 
                color='k', fontsize=20, fontweight='bold', 
                bbox=dict(boxstyle='round', facecolor='w', alpha=1))

    ax.set_xlabel('Zeroed Variable', fontsize=22)
    ax.set_ylabel('$R^2$ Performance', fontsize=22)
    ax.set_xticklabels(labels=['CWV', 'Z500', 'TAS'], fontsize=15)
    ax.set_yticks([-0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0], [-0.4, -0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0], fontsize=15)
    ax.set_ylim(-0.5, 1.1)

    if num2 == 0:
        #ax.set_ylabel(f'', fontsize=27, fontweight='bold')
        ax.annotate(f'Nonlinear CNNs', xy=(-.37,.2), xycoords='axes fraction', 
                        fontsize=25, rotation='vertical', fontweight='bold')

# --- PART 3: Bounding Boxes (Auto-adjusting) ---
fig.canvas.draw()

# CESM Rectangle (First 3 plots)
cesm_bbox = Bbox.union([ax.get_position() for ax in axes[:3]])
rect_cesm = patches.Rectangle((cesm_bbox.x0 - 0.09, cesm_bbox.y0 - 0.035),
                              cesm_bbox.width + 0.11, cesm_bbox.height + 0.07,
                              lw=4, ec='lightgrey', fc='none', transform=fig.transFigure, zorder=1000)
fig.text(cesm_bbox.x1 + 0.02, cesm_bbox.y1 + 0.02, 'CESM2-LE Tested', fontsize=37, va='top', ha='left', weight='bold', rotation=270)
fig.patches.append(rect_cesm)

# ERA5 Rectangle (Everything from index 3 to 12)
era5_bbox = Bbox.union([ax.get_position() for ax in axes[3:]])
rect_era5 = patches.Rectangle((era5_bbox.x0 - 0.09, era5_bbox.y0 + 0.17),
                              era5_bbox.width + 0.11, era5_bbox.height - 0.1435,
                              lw=4, ec='lightgrey', fc='none', transform=fig.transFigure, zorder=1000)
fig.text(era5_bbox.x1 + 0.02, era5_bbox.y1 - 0.14, 'ERA5 Tested', fontsize=37, va='top', ha='left', weight='bold', rotation=270)
fig.patches.append(rect_era5)

#CDF Rectangle (Final row)
cdf_bbox = Bbox.union([ax.get_position() for ax in axes[3:]])
rect_cdf = patches.Rectangle((era5_bbox.x0 - 0.09, era5_bbox.y0 - 0.02),
                              era5_bbox.width + 0.11, era5_bbox.height - 0.402,
                              lw=4, ec='lightgrey', fc='none', transform=fig.transFigure, zorder=1000)
fig.text(era5_bbox.x1 + 0.02, era5_bbox.y1 - 0.45, 'Feature Omission', fontsize=37, va='top', ha='left', weight='bold', rotation=270)
fig.patches.append(rect_cdf)

plt.savefig('cdfs_labeled.png', format='png', dpi=500, bbox_inches='tight')